# Myanmar Subword Tokenization Demo
**BPE vs Unigram with SentencePiece**

This notebook demonstrates the academic mini-project pipeline:
preprocessing → training (optional) → tokenization samples → evaluation metrics → graphs → error analysis.


## 0. Setup


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


## 1. Peek at the corpus


In [ ]:
raw = PROJECT_ROOT / "data" / "raw" / "myanmar_corpus.txt"
cleaned = PROJECT_ROOT / "data" / "processed" / "cleaned_corpus.txt"
train = PROJECT_ROOT / "data" / "processed" / "train_corpus.txt"
evalp = PROJECT_ROOT / "data" / "processed" / "eval_corpus.txt"

for p in [raw, cleaned, train, evalp]:
    if p.exists():
        n = sum(1 for _ in open(p, encoding="utf-8"))
        print(f"{p.relative_to(PROJECT_ROOT)}: {n} lines")
    else:
        print(f"MISSING {p}")

print("\nSample cleaned lines:")
if cleaned.exists():
    for i, line in enumerate(open(cleaned, encoding="utf-8")):
        print(line.rstrip())
        if i >= 4:
            break


## 2. Load trained models and tokenize demo sentences


In [ ]:
import sentencepiece as spm

MODELS = PROJECT_ROOT / "models"
SENTENCES = [
    "မြန်မာနိုင်ငံသည် ဖွံ့ဖြိုးတိုးတက်လာနေသည်။",
    "ကျန်းမာရေးဝန်ကြီးဌာနသည် ပြည်သူများအတွက် ဝန်ဆောင်မှုပေးနေသည်။",
]

def load(name):
    sp = spm.SentencePieceProcessor()
    model_path = MODELS / name
    if not sp.LoadFromSerializedProto(model_path.read_bytes()):
        raise ValueError(f"Failed to load model: {model_path}")
    return sp

vocab = 8000
bpe = load(f"bpe_{vocab}.model")
uni = load(f"unigram_{vocab}.model")

for s in SENTENCES:
    print("=" * 60)
    print("Original :", s)
    print("BPE      :", " | ".join(bpe.encode(s, out_type=str)))
    print("Unigram  :", " | ".join(uni.encode(s, out_type=str)))


## 3. Evaluation table


In [ ]:
import pandas as pd

csv_path = PROJECT_ROOT / "results" / "comparison.csv"
df = pd.read_csv(csv_path)
df


## 4. Visualization


In [ ]:
from IPython.display import Image, display

graphs = PROJECT_ROOT / "results" / "graphs"
for name in [
    "compression_ratio.png",
    "oov_rate.png",
    "vocab_size_effect.png",
    "token_count_comparison.png",
]:
    path = graphs / name
    print(name)
    display(Image(filename=str(path)))


## 5. Morphology error analysis


In [ ]:
from src.evaluation import error_analysis

findings = error_analysis(MODELS, vocab_size=8000)
for item in findings:
    print(f"Word: {item['word']}")
    print(f"  BPE     ({item['bpe_count']}): {' | '.join(item['bpe_tokens'])}")
    print(f"  Unigram ({item['unigram_count']}): {' | '.join(item['unigram_tokens'])}")
    print(f"  → {item['verdict']}\n")


## 6. Optional: re-run full pipeline

Uncomment to regenerate everything (takes a few minutes).


In [ ]:
# from src.run_pipeline import run_all
# run_all(skip_generate=False)


## Takeaways

- Larger vocabulary → higher compression, longer average tokens.
- BPE compresses slightly more than Unigram at the same vocab size.
- Unigram often preserves Myanmar compounds / loanwords better in qualitative checks.
